# Notebook 04: Skill Knowledge Graph & Community Analysis
## SkillGraph AI - NetworkX Graph Construction & Centrality Metrics

This notebook constructs the bipartite and co-occurrence skill knowledge graph, calculates centrality metrics, detects communities with greedy modularity, and extracts targeted career subgraphs.

In [ ]:
import sqlite3
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

conn = sqlite3.connect("../data/skillgraph.db")
edges_df = pd.read_sql_query("SELECT skill_id_1, skill_id_2, weight FROM skill_relationships WHERE weight > 2", conn)
skills_df = pd.read_sql_query("SELECT id, canonical_name, category FROM skills", conn).set_index('id')
print(f"Loaded {len(edges_df)} significant relationships.")

### 1. Build NetworkX Graph

In [ ]:
G = nx.Graph()
for _, row in edges_df.iterrows():
    u = int(row['skill_id_1'])
    v = int(row['skill_id_2'])
    w = float(row['weight'])
    if u in skills_df.index and v in skills_df.index:
        G.add_edge(skills_df.loc[u, 'canonical_name'], skills_df.loc[v, 'canonical_name'], weight=w)

print(f"Graph nodes: {G.number_of_nodes()}, Graph edges: {G.number_of_edges()}")
print(f"Graph density: {nx.density(G):.4f}")

### 2. Centrality Analysis

In [ ]:
degree_cent = nx.degree_centrality(G)
top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]

top_degree_df = pd.DataFrame(top_degree, columns=["Skill", "Degree Centrality"])
display(top_degree_df)

plt.figure(figsize=(10, 4))
plt.barh(top_degree_df['Skill'][::-1], top_degree_df['Degree Centrality'][::-1], color='#0d9488')
plt.title("Top 10 Central Hub Skills in Knowledge Graph")
plt.xlabel("Degree Centrality")
plt.tight_layout()
plt.show()

### 3. Community Detection & Visualization

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))
print(f"Detected {len(communities)} modular skill clusters.")

for i, comm in enumerate(communities[:4]):
    print(f"Cluster {i+1} ({len(comm)} skills): {list(comm)[:8]}...")
conn.close()